# MATH-F on Colab

Runs the MATH-F evaluation: a small Lean 4 theorem-proving model attempts miniF2F-style problems, with the real Lean toolchain as the sole correctness authority and up to 5 bounded repair attempts per problem.

**Before running the overnight cell:** Runtime -> Change runtime type -> GPU (A100 if available).

Steps in this notebook:
1. Clone the repo and install Python dependencies
2. Check the GPU
3. Install the Lean 4 / Lake toolchain and a Mathlib-enabled project (this is the slow step -- ~15-30 min the first time)
4. Configure the model
5. Run `smoke-test` (fast sanity check)
6. Run a small trial evaluation
7. Review the configuration, then launch the full overnight evaluation
8. Resume after a disconnect, inspect results, and download them

## 1. Clone and install

In [ ]:
# Replace with your fork/clone URL once pushed to GitHub.
REPO_URL = "https://github.com/<your-username>/math-f.git"

!git clone "$REPO_URL" /content/math-f
%cd /content/math-f
!pip install -q -r requirements.txt

## 2. GPU check

In [ ]:
!nvidia-smi
import torch
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("bf16 supported:", torch.cuda.is_bf16_supported())

## 3. Install the Lean 4 / Lake toolchain + Mathlib

This creates a small Lake project that depends on Mathlib and downloads Mathlib's prebuilt `.olean` cache (via `lake exe cache get`) instead of compiling Mathlib from source, which would take hours. This is the slow cell -- expect ~15-30 minutes.

In [ ]:
# Install elan (the Lean version manager) and put it on PATH.
!curl https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -sSf | sh -s -- -y
import os
os.environ["PATH"] = f"{os.path.expanduser('~/.elan/bin')}:{os.environ['PATH']}"
!elan --version
!lake --version

In [ ]:
%%bash
set -e
mkdir -p /content/lean_project
cd /content/lean_project
if [ ! -f lakefile.toml ] && [ ! -f lakefile.lean ]; then
  lake +leanprover/lean4:stable new mathlib_project math
  mv mathlib_project/* mathlib_project/.* . 2>/dev/null || true
fi
cat > lakefile.toml << 'EOF'
name = "math_f_verify"
defaultTargets = ["MathFVerify"]

[[require]]
name = "mathlib"
git = "https://github.com/leanprover-community/mathlib4"

[[lean_lib]]
name = "MathFVerify"
EOF
mkdir -p MathFVerify
echo "-- placeholder" > MathFVerify/Basic.lean
lake update
lake exe cache get
lake build

In [ ]:
LEAN_PROJECT_DIR = "/content/lean_project"
os.environ["LEAN_PROJECT_DIR"] = LEAN_PROJECT_DIR
!cd $LEAN_PROJECT_DIR && lake env lean --version

## 4. Configure the model

In [ ]:
# The default is a real, current ~1.5B Lean 4 theorem-proving specialist.
# Override with any other Lean-capable causal LM checkpoint on the Hub.
os.environ["MODEL_NAME"] = "AI-MO/Kimina-Prover-Preview-Distill-1.5B"
os.environ["MODEL_BACKEND"] = "hf"
os.environ["MAX_ATTEMPTS"] = "5"
print("MODEL_NAME:", os.environ["MODEL_NAME"])

## 5. Smoke test

Fast infrastructure check: Python/PyTorch/CUDA/GPU/Transformers, model + tokenizer loading, one real generation, the deterministic cleaner, one real Lean verification, and the trajectory writer round-trip.

In [ ]:
!python -m math_f smoke-test --backend hf --lean-project-dir "$LEAN_PROJECT_DIR"

## 6. Small trial evaluation (a handful of real miniF2F tasks)

In [ ]:
!python -m math_f evaluate \
  --split valid \
  --limit 5 \
  --max-attempts 5 \
  --lean-project-dir "$LEAN_PROJECT_DIR" \
  --output runs/trial_run

## 7. Review configuration before the overnight run

Confirm GPU, model, task count, and max attempts before launching a long run.

In [ ]:
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")
print("CUDA:", torch.version.cuda if torch.cuda.is_available() else "n/a")
print("Model:", os.environ["MODEL_NAME"])
print("Max attempts:", os.environ["MAX_ATTEMPTS"])
!python -c "from math_f.datasets.minif2f import load_tasks; print('Test split size:', len(load_tasks('AI-MO/minif2f_test', 'test')))"

## 8. Overnight evaluation

This is the long-running cell. It checkpoints every attempt to disk immediately, so it is safe to stop (Ctrl+C / interrupt the cell) and resume later with the cell in section 9.

In [ ]:
!python -m math_f evaluate \
  --split test \
  --max-attempts 5 \
  --lean-project-dir "$LEAN_PROJECT_DIR" \
  --output runs/experiment_001

## 9. Resume after a disconnect

If the Colab runtime dies partway through, reconnect, re-run cells 1-4, and run this cell instead of section 8. It reuses the run's stored configuration and skips every task that already has a final trajectory record.

In [ ]:
!python -m math_f resume --input runs/experiment_001

## 10. Inspect results

In [ ]:
!python -m math_f stats --input runs/experiment_001 --json

## 11. Download the raw trajectories and reports

In [ ]:
!zip -qr runs_experiment_001.zip runs/experiment_001
from google.colab import files
files.download("runs_experiment_001.zip")